# Discover and verify an agent Skill

Browse the public Skills catalogue, then verify the official Super ii Skill manifest, file hashes, and Ed25519 signature before setup. Discovery never means automatic installation or trust.

In [ ]:
import base64
import hashlib
import json
from urllib.request import Request, urlopen

ORIGIN = "https://superii.site"

def get_bytes(path):
    request = Request(f"{ORIGIN}{path}", headers={"Accept": "application/json, text/plain"})
    with urlopen(request, timeout=20) as response:
        return response.read()

catalog = json.loads(get_bytes("/api/skills"))
print(f"Catalogue version: {catalog['version']} · entries: {len(catalog['skills'])}")
for skill in catalog["skills"][:10]:
    print(f"- {skill['name']} [{skill['category']}]")

## Verify the signed official manifest

Install `cryptography` in the notebook environment. Verification uses the exact downloaded UTF-8 bytes and checks the declared key identifier before validating every file checksum.

In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey

base = "/skills/superii"
manifest_bytes = get_bytes(f"{base}/manifest.json")
manifest = json.loads(manifest_bytes)
signature = json.loads(get_bytes(f"{base}/signature.json"))
public_key = json.loads(get_bytes(f"{base}/public-key.json"))

if signature["key_id"] != public_key["key_id"]:
    raise RuntimeError("Manifest key identifier mismatch")
key_bytes = base64.b64decode(public_key["public_key"], validate=True)
if hashlib.sha256(key_bytes).hexdigest() != public_key["sha256"]:
    raise RuntimeError("Public-key checksum mismatch")
Ed25519PublicKey.from_public_bytes(key_bytes).verify(
    base64.b64decode(signature["signature"], validate=True), manifest_bytes
)
print(f"Signature verified with {signature['key_id']}")

In [ ]:
for item in manifest["files"]:
    payload = get_bytes(f"{base}/{item['path']}")
    actual = hashlib.sha256(payload).hexdigest()
    if actual != item["sha256"]:
        raise RuntimeError(f"Checksum mismatch: {item['path']}")
    print(f"verified {item['path']}")

## Review before setup

Read `SKILL.md` and every referenced instruction, confirm the requested permissions and destinations, and install only through a visible agent control you trust. This notebook verifies origin and bytes; it does not decide whether the Skill is appropriate for your task.